PART 03：循环只加一行——拒绝也是一种输入
闸门造好了，装在哪？看拼接的代码：

In [ ]:
def check_permission(block) -> bool:
    # 闸门 1: 硬拒绝(只管 bash,黑名单里全是 shell 命令)
    if block.name == "bash":
        reason = check_deny_list(block.input.get("command", ""))
        if reason:
            print(f"\n\033[31m[blocked] {reason}\033[0m")
            return False

    # 闸门 2 + 3: 规则匹配 → 命中就问人
    reason = check_rules(block.name, block.input)
    if reason:
        decision = ask_user(block.name, block.input, reason)
        if decision == "deny":
            return False

    return True

然后是整个第三篇对主循环的全部改动——注意，是全部：

In [ ]:
for block in tool_calls:
    print(f"> {block.name}")

    if not check_permission(block):                    # ← 新增的两行
        results.append({"type": "tool_result",
                        "tool_use_id": block.id,
                        "content": "Permission denied."})
        continue

    handler = TOOL_HANDLERS.get(block.name)            # 以下全是旧代码
    output = handler(**block.input) if handler else f"Unknown: {block.name}"
    results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})

第一篇造的心脏，第三篇依然一行没动。工具在变、知识在变、权限在变，循环不变——这个母题到这一篇应该已经刻进你脑子里了。

真正值得盯的是新增那两行里的一个决定：拦下一个调用时，我们没抛异常，没退出程序，而是往账本里记了一笔普通的 tool_result，内容是 "Permission denied."。

想想这意味着什么。拒绝信息会作为 user 消息回流给模型，下一圈循环，模型会读到自己被拒了，然后自己决定怎么办：换条路？问用户？还是放弃这个方案？循环照转，agent 不死。

对比一下另一种直觉写法：检查不过就 raise PermissionError 然后崩掉。那是"程序"的思路——出错了，停。但 agent 不是程序，agent 的每一次意外，都应该变成它的输入，而不是它的终止。报错是模型的眼睛，这个第一篇就讲过；现在补上后半句：拒绝也是。

而且这个设计悄悄解决了"拦了之后卡死"的问题。你 deny 了模型删文件的请求，它不会停在原地干等——它收到了一条明确的反馈，游戏继续。

好，门装好了，账本会记下每一次拒绝。接下来跑起来，看看这只装了门禁的 Agent，行为有什么变化——以及，它怎么应对被拒。

![](执行流程.png)